# Talking to MySQL Workbench using Local LLMs
### LangChain + Ollama + MySQL

## Imports and settings

In [11]:
from langchain_community.utilities import SQLDatabase
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


## Connect to the database

###  LangChain SQLDatabase Functions 
Link: https://reference.langchain.com/python/langchain-community/utilities/sql_database/SQLDatabase

In [12]:
mysql_uri = "mysql+mysqlconnector://[database user name]:[password for you database connection]@[host server (localhost)]/[database name]"
db = SQLDatabase.from_uri(mysql_uri)
print("Connected. Dialect:", db.dialect)

Connected. Dialect: mysql


In [13]:
print(db.get_usable_table_names())

['courses', 'enrollments', 'students']


In [14]:
print(db.get_table_info())


CREATE TABLE courses (
	course_id INTEGER NOT NULL, 
	course_name VARCHAR(100) NOT NULL, 
	department VARCHAR(50) NOT NULL, 
	credits INTEGER NOT NULL, 
	PRIMARY KEY (course_id)
)ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4

/*
3 rows from courses table:
course_id	course_name	department	credits
101	Intro to Programming	Computer Science	3
102	Databases	Computer Science	4
103	Calculus I	Mathematics	4
*/


CREATE TABLE enrollments (
	enrollment_id INTEGER NOT NULL AUTO_INCREMENT, 
	student_id INTEGER NOT NULL, 
	course_id INTEGER NOT NULL, 
	semester VARCHAR(20) NOT NULL, 
	score DECIMAL(5, 2) NOT NULL, 
	PRIMARY KEY (enrollment_id), 
	CONSTRAINT enrollments_ibfk_1 FOREIGN KEY(student_id) REFERENCES students (student_id), 
	CONSTRAINT enrollments_ibfk_2 FOREIGN KEY(course_id) REFERENCES courses (course_id)
)ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4

/*
3 rows from enrollments table:
enrollment_id	student_id	course_id	semester	score
1	1	101	Fall 

In [16]:
print(db.run("select * from students limit 1;"))

[(1, 'Alice', 'Johnson', 'alice.johnson@school.edu', 'Computer Science', 2023)]


In [17]:
db.run("select s.student_id, s.first_name, c.course_name, e.score from enrollments e join students s on e.student_id= s.student_id join courses c on c.course_id=e.course_id order by e.score desc;")

"[(12, 'Liam', 'Intro to Programming', Decimal('96.00')), (3, 'Carla', 'Calculus I', Decimal('95.00')), (7, 'Grace', 'Genetics', Decimal('94.00')), (3, 'Carla', 'Linear Algebra', Decimal('93.50')), (12, 'Liam', 'Databases', Decimal('93.00')), (1, 'Alice', 'Intro to Programming', Decimal('92.50')), (2, 'Bob', 'Databases', Decimal('91.00')), (5, 'Emma', 'Intro to Programming', Decimal('90.00')), (6, 'Farid', 'Linear Algebra', Decimal('90.00')), (9, 'Isabel', 'Genetics', Decimal('89.00')), (6, 'Farid', 'Calculus I', Decimal('88.50')), (1, 'Alice', 'Databases', Decimal('88.00')), (12, 'Liam', 'Linear Algebra', Decimal('87.00')), (10, 'Jack', 'Databases', Decimal('86.50')), (2, 'Bob', 'Intro to Programming', Decimal('85.00')), (5, 'Emma', 'Databases', Decimal('84.00')), (8, 'Hassan', 'Classical Mechanics', Decimal('82.00')), (3, 'Carla', 'Intro to Programming', Decimal('81.00')), (9, 'Isabel', 'Calculus I', Decimal('80.50')), (1, 'Alice', 'Calculus I', Decimal('79.00')), (4, 'David', 'Class

## Connect to Ollama
https://reference.langchain.com/python/langchain-ollama/llms/OllamaLLM

Ollama runs the model locally and listens on port `11434`. 

echo %OLLAMA_HOST%

In [18]:
llm = OllamaLLM(
    model= "llama3.1",
    base_url="http://localhost:11434",
    temperature=0,
)

In [20]:
llm.invoke("what is sql")

"SQL (Structured Query Language) is a programming language designed for managing and manipulating data stored in relational database management systems (RDBMS). It's a standard language for accessing, managing, and modifying data in relational databases.\n\nSQL is used to perform various operations on data, such as:\n\n1. **Creating and modifying database structures**: Creating tables, indexes, views, and relationships between tables.\n2. **Inserting, updating, and deleting data**: Adding new data, modifying existing data, and deleting data from tables.\n3. **Querying data**: Retrieving specific data from tables using SELECT statements.\n4. **Managing database security**: Controlling access to data and database objects using user accounts and permissions.\n\nSQL is a declarative language, meaning you specify what you want to do with your data, rather than how to do it. This makes it easier to write and maintain database applications.\n\nSome common SQL commands include:\n\n* `SELECT`: 

## Chain 1, question -> SQL

In [24]:
PROMPT1 = ChatPromptTemplate.from_template(
    """Based on the table schema below, write a MySQL query that answers the user's question.

{schema}

Question: {question}

Return only the SQL query. No explanation, no markdown.
SQL Query:"""
)

In [22]:
# example string formatting 
print("my name is {name}".format(name="Aziz"))
PROMPT1.format(schema = db.get_table_info())

my name is Aziz


In [25]:
# example prompt format
PROMPT1.format(schema = db.get_table_info(), question= "get me all students?")

"Human: Based on the table schema below, write a MySQL query that answers the user's question.\n\n\nCREATE TABLE courses (\n\tcourse_id INTEGER NOT NULL, \n\tcourse_name VARCHAR(100) NOT NULL, \n\tdepartment VARCHAR(50) NOT NULL, \n\tcredits INTEGER NOT NULL, \n\tPRIMARY KEY (course_id)\n)ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4\n\n/*\n3 rows from courses table:\ncourse_id\tcourse_name\tdepartment\tcredits\n101\tIntro to Programming\tComputer Science\t3\n102\tDatabases\tComputer Science\t4\n103\tCalculus I\tMathematics\t4\n*/\n\n\nCREATE TABLE enrollments (\n\tenrollment_id INTEGER NOT NULL AUTO_INCREMENT, \n\tstudent_id INTEGER NOT NULL, \n\tcourse_id INTEGER NOT NULL, \n\tsemester VARCHAR(20) NOT NULL, \n\tscore DECIMAL(5, 2) NOT NULL, \n\tPRIMARY KEY (enrollment_id), \n\tCONSTRAINT enrollments_ibfk_1 FOREIGN KEY(student_id) REFERENCES students (student_id), \n\tCONSTRAINT enrollments_ibfk_2 FOREIGN KEY(course_id) REFERENCES courses (course_id)\n)ENGINE=InnoDB

In [27]:
def get_schema(_):
    return db.get_table_info()


def clean_sql(text):
    text = text.strip()
    if text.startswith("```"):
        temp = text.split("\n", 1)
        text = text.split("\n", 1)[1] if "\n" in text else text[3:]
    if text.endswith("```"):
        text = text.rsplit("```", 1)[0]
    return text.strip()

# Quick test of the cleaner
clean_sql("```sql\nSELECT 1;\n```")

'SELECT 1;'

In [28]:
#define sql chain 
# fetch schema -> give to prompt -> give to llm -> parse output -> clean output 
sql_chain=(
    RunnablePassthrough.assign(schema = get_schema)
    | PROMPT1
    | llm
    | StrOutputParser()
    | RunnableLambda(clean_sql)
)

In [29]:
sql_chain.invoke({"question": "How many students are there?"})

'SELECT COUNT(student_id) FROM students'

In [30]:
sql_chain.invoke({"question": "What courses did Alice Johnson take, and what were her scores?"})

"SELECT c.course_name, e.score FROM enrollments e INNER JOIN courses c ON e.course_id = c.course_id INNER JOIN students s ON e.student_id = s.student_id WHERE s.first_name = 'Alice' AND s.last_name = 'Johnson';"

In [31]:
sql_chain.invoke({"question": "What is the average score for each course, highest first?"})

'SELECT course_id, AVG(score) AS average_score FROM enrollments GROUP BY course_id ORDER BY average_score DESC'

In [32]:
q = sql_chain.invoke({"question": "How many students are in each major?"})
print(q)
print(db.run(q))

SELECT major, COUNT(student_id) as num_students FROM students GROUP BY major
[('Computer Science', 5), ('Mathematics', 3), ('Physics', 2), ('Biology', 2)]


## Chain 2, SQL result -> English answer



In [33]:
ANSWER_PROMPT = ChatPromptTemplate.from_template(
    """You are answering a question using the result of a SQL query.

Question: {question}
SQL Query: {query}
SQL Result: {response}

Write a clear answer in one or two sentences. Do not show SQL. Do not explain how you got it.
Answer:"""
)

In [34]:
qa_chain = (
    RunnablePassthrough.assign(query=sql_chain)
    .assign(response=lambda v: db.run(v["query"]))
    | ANSWER_PROMPT
    | llm
    | StrOutputParser()
)

1. assign(query=sql_chain) runs Chain 1 and stores the SQL under `query`

2. assign(response=...) runs that SQL on MySQL and stores the result under `response`

3. ANSWER_PROMPT | llm turns question + SQL + result into English

### Question to answer

In [35]:
qa_chain.invoke({"question": "How many students are there?"})

'There are 12 students.'

In [36]:
qa_chain.invoke({"question": "Which course has the most enrollments, state course name?"})

'The course with the most enrollments is "Intro to Programming".'

In [37]:
qa_chain.invoke({"question": "Who are the top 3 students by average score?"})

'The top 3 students by average score are Liam Chen with an average score of 92, Carla Gomez with an average score of 89.83, and Farid Khan with an average score of 89.25.'

## Prompt is very Important

In [38]:
PROMPT = ChatPromptTemplate.from_template(
"""Based on the table schema below, write a MySQL query that answers the user's question.

{schema}

Question: {question}"""
)

chain = (
    RunnablePassthrough.assign(schema=get_schema)
    | PROMPT
    | llm
    | StrOutputParser()
    | RunnableLambda(clean_sql)
)

chain.invoke({"question": "Give me total number of students?"})

"Here is the MySQL query that answers the user's question:\n\n```sql\nSELECT COUNT(*) AS total_students\nFROM students;\n```\n\nThis query uses the `COUNT(*)` function to count the number of rows in the `students` table, and assigns the result to a column named `total_students`. The `AS` keyword is used to give an alias to the column."

### Hallucination

In [39]:
sql_chain.invoke({"question": "What is the average age of our students?"})

'SELECT AVG(CURDATE() - enrollment_year) AS average_age FROM students'